In [ ]:
# imports
import pandas as pd
import numpy as np
import scipy as sp

import matplotlib.pyplot as plt
import seaborn as sns

from typing import Callable, Optional, List, Self
from numpy.typing import NDArray

from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.linear_model import RANSACRegressor

In [ ]:
from kernel_regression_utils import KernelRegressor
from kernel_regression_utils import lKernelType, lH

In [ ]:
# defining of features matrix
features_df = pd.read_csv('data/features_with_pb.csv')
features_df = features_df.loc[:, 'CID':]
mX = features_df.loc[:, 'f1_mass_PC1':].to_numpy()
print(mX.shape)

In [ ]:
data_df = pd.read_csv('data/waka_dragon_merged.csv')
data_df.head()

In [ ]:
# defining of target matrix
vY = data_df['Imax'].astype(float).values
print(vY.shape)

In [ ]:
numComb = len(lKernelType) * len(lH)
dData = {
    'Kernel Type': [],
    'h' : [],
    'R2': [0.0] * numComb
} # creation of dictionary

for ii, kernelType in enumerate(lKernelType):
    for jj, paramH in enumerate(lH):
        dData['Kernel Type'].append(kernelType)
        dData['h'].append(paramH)

dfModelScore = pd.DataFrame(data=dData)
dfModelScore

In [ ]:
for ii in range(numComb):
    kernelType = dfModelScore.loc[ii, 'Kernel Type']
    paramH = dfModelScore.loc[ii, 'h']

    print(f'Processing model {ii + 1:03d} out of {numComb} with `Kernel Type` = {kernelType} and `h` = {paramH}.')

    oKerReg = KernelRegressor(kernelType = kernelType, paramH = paramH)
    
    vYPred = cross_val_predict(oKerReg, mX, vY, cv = KFold(n_splits = mX.shape[0]))

    scoreR2 = r2_score(vY, vYPred)
    dfModelScore.loc[ii, 'R2'] = scoreR2
    print(f'Finished processing model {ii + 1:03d} with `R2 = {scoreR2}.')


In [ ]:
dfModelScore.sort_values(by='R2', ascending=False).head(10)

In [ ]:
# Trial to implement RANSAC
best_mae = np.inf
best_ratio = None

for ratio in np.arange(0.35, 1.00, 0.05):

    ransac = RANSACRegressor(
        estimator=KernelRegressor(
            kernelType='Gaussian',
            paramH=5.98
        ),
        min_samples=ratio,
        max_trials=1000,
        random_state=42
    )

    ransac.fit(mX, vY)

    inlier_mask = ransac.inlier_mask_

    mX_clean = mX[inlier_mask]
    vY_clean = vY[inlier_mask]

    vYPred_clean = cross_val_predict(KernelRegressor(kernelType='Gaussian', paramH=5.98), mX_clean, vY_clean, cv=KFold(n_splits=len(vY_clean), shuffle=True, random_state=42))
    mae = mean_absolute_error(vY_clean, vYPred_clean)
    r2 = r2_score(vY_clean, vYPred_clean)
    

    print(f"ratio={ratio:.2f}, MAE={mae:.3f}")
    print(f"ratio={ratio:.2f}, R2={r2:.3f}")

    if mae < best_mae:
        best_mae = mae
        best_ratio = ratio

print(f"\nBest ratio={best_ratio:.2f}")
print(f"Best MAE={best_mae:.3f}")